In [1]:
import os
import json
import re
import math
import random
import copy
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix, classification_report, f1_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

In [2]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

# ── Output directories ──────────────────────────────────────────────────────
os.makedirs('embeddings', exist_ok=True)
os.makedirs('models',     exist_ok=True)
os.makedirs('data',       exist_ok=True)

print('Directories ready.')

Using device: cpu
Directories ready.


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# UTILITY: Corpus loader — returns list-of-sentences (each sentence = list of tokens)
# ─────────────────────────────────────────────────────────────────────────────
def load_corpus(path, lowercase=True):
    """Read a text file; return a list of token lists (one per non-empty line)."""
    sentences = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if lowercase:
                line = line.lower()
            sentences.append(line.split())
    return sentences


# ─────────────────────────────────────────────────────────────────────────────
# UTILITY: Build vocabulary capped at max_vocab most-frequent tokens
# Returns: word2idx, idx2word, token frequency counter
# ─────────────────────────────────────────────────────────────────────────────
def build_vocab(sentences, max_vocab=10_000, specials=('<PAD>', '<UNK>')):
    """Build word→index mapping; all rare words map to <UNK>."""
    counter = Counter(tok for sent in sentences for tok in sent)
    most_common = [w for w, _ in counter.most_common(max_vocab)]

    word2idx = {s: i for i, s in enumerate(specials)}
    for word in most_common:
        if word not in word2idx:
            word2idx[word] = len(word2idx)

    idx2word = {i: w for w, i in word2idx.items()}
    return word2idx, idx2word, counter


# ─────────────────────────────────────────────────────────────────────────────
# UTILITY: Cosine similarity between two 1-D numpy arrays
# ─────────────────────────────────────────────────────────────────────────────
def cosine_similarity(a, b):
    a = a / (np.linalg.norm(a) + 1e-9)
    b = b / (np.linalg.norm(b) + 1e-9)
    return float(np.dot(a, b))


# ─────────────────────────────────────────────────────────────────────────────
# UTILITY: Top-N nearest neighbours from an embedding matrix
# ─────────────────────────────────────────────────────────────────────────────
def nearest_neighbours(query_word, embeddings, word2idx, idx2word, top_n=10):
    """Return top_n nearest neighbours (excluding the query word itself)."""
    if query_word not in word2idx:
        print(f'  "{query_word}" not in vocabulary.')
        return []
    idx  = word2idx[query_word]
    vec  = embeddings[idx]
    # Normalise entire matrix once
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-9
    normed = embeddings / norms
    sims  = normed @ (vec / (np.linalg.norm(vec) + 1e-9))
    sims[idx] = -1          # exclude self
    top_ids = np.argsort(sims)[::-1][:top_n]
    return [(idx2word[i], float(sims[i])) for i in top_ids]


print('Utilities loaded.')

Utilities loaded.


In [4]:
# LOAD CORPUS AND BUILD TERM-DOCUMENT MATRIX

CLEANED_PATH = 'cleaned.txt'
RAW_PATH     = 'raw.txt'
META_PATH    = 'metadata.json'

sentences_clean = load_corpus(CLEANED_PATH)
print(f'Loaded {len(sentences_clean)} sentences from cleaned.txt')

word2idx, idx2word, freq_counter = build_vocab(sentences_clean, max_vocab=10_000)
VOCAB_SIZE = len(word2idx)
print(f'Vocabulary size (incl. specials): {VOCAB_SIZE}')

# Save word2idx for later use
with open('embeddings/word2idx.json', 'w', encoding='utf-8') as f:
    json.dump(word2idx, f, ensure_ascii=False)
print('word2idx saved to embeddings/word2idx.json')

Loaded 24241 sentences from cleaned.txt
Vocabulary size (incl. specials): 10002
word2idx saved to embeddings/word2idx.json


In [5]:
# COMPUTE TF-IDF MATRIX

N_DOCS = len(sentences_clean)

# ── Step 1: Document-frequency vector df[w] ─────────────────────────────────
df = np.zeros(VOCAB_SIZE, dtype=np.float32)
for sent in sentences_clean:
    # Unique vocabulary indices in this document
    idxs = set(word2idx.get(tok, word2idx['<UNK>']) for tok in sent)
    for i in idxs:
        df[i] += 1

# ── Step 2: IDF vector ───────────────────────────────────────────────────────
idf = np.log(N_DOCS / (1 + df))        # shape: (VOCAB_SIZE,)

# ── Step 3: Build sparse TF-IDF matrix using a list-of-dicts for efficiency ──
# We store the full dense matrix only for the top vocabulary; adjust if RAM-limited.
tfidf_matrix = np.zeros((N_DOCS, VOCAB_SIZE), dtype=np.float32)

for d_idx, sent in enumerate(sentences_clean):
    if len(sent) == 0:
        continue
    token_ids = [word2idx.get(tok, word2idx['<UNK>']) for tok in sent]
    tf_counts  = Counter(token_ids)
    sent_len   = len(token_ids)
    for w_idx, cnt in tf_counts.items():
        tf = cnt / sent_len
        tfidf_matrix[d_idx, w_idx] = tf * idf[w_idx]

np.save('embeddings/tfidf_matrix.npy', tfidf_matrix)
print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print('Saved → embeddings/tfidf_matrix.npy')

TF-IDF matrix shape: (24241, 10002)
Saved → embeddings/tfidf_matrix.npy


In [6]:
# Load metadata
with open(META_PATH, 'r', encoding='utf-8') as f:
    metadata = json.load(f)

print(f'Loaded metadata for {len(metadata)} documents')


# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Define automatic topic classifier from title
# ─────────────────────────────────────────────────────────────────────────────
def infer_topic(title):
    title = title.lower()

    if any(k in title for k in [
        'پی آئی اے', 'نجکاری', 'حصص', 'معیشت', 'روپے',
        'کمپنی', 'بینک', 'سٹاک', 'بزنس', 'اقتصاد'
    ]):
        return 'Business'

    elif any(k in title for k in [
        'عمران خان', 'پی ٹی آئی', 'فوج', 'حکومت',
        'وزیراعظم', 'الیکشن', 'پارلیمان', 'سیاسی'
    ]):
        return 'Politics'

    elif any(k in title for k in [
        'کرکٹ', 'فٹبال', 'میچ', 'ٹورنامنٹ',
        'ورلڈ کپ', 'کھلاڑی'
    ]):
        return 'Sports'

    elif any(k in title for k in [
        'قتل', 'پولیس', 'عدالت', 'مقدمہ',
        'جیل', 'گرفتار'
    ]):
        return 'Crime'

    elif any(k in title for k in [
        'ٹیکنالوجی', 'گوگل', 'ایپل', 'مصنوعی ذہانت',
        'موبائل', 'انٹرنیٹ'
    ]):
        return 'Technology'

    else:
        return 'General'


# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Create document-topic mapping
# ─────────────────────────────────────────────────────────────────────────────
doc_topics = []

for i in range(1, N_DOCS + 1):
    key = str(i)

    if key in metadata:
        title = metadata[key]['title']
        topic = infer_topic(title)
    else:
        topic = 'General'

    doc_topics.append(topic)

print('Topic distribution:')
print(Counter(doc_topics))


# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: Compute top TF-IDF words per topic
# ─────────────────────────────────────────────────────────────────────────────
topic_to_docs = defaultdict(list)

for doc_id, topic in enumerate(doc_topics):
    topic_to_docs[topic].append(doc_id)


top_words_per_topic = {}

for topic, doc_ids in topic_to_docs.items():

    # Mean TF-IDF vector for all docs in this topic
    mean_scores = tfidf_matrix[doc_ids].mean(axis=0)

    # Ignore PAD token
    mean_scores[word2idx['<PAD>']] = 0

    # Top 10 indices
    top_ids = np.argsort(mean_scores)[::-1][:10]

    top_words = [(idx2word[i], float(mean_scores[i])) for i in top_ids]

    top_words_per_topic[topic] = top_words


# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: Print Results
# ─────────────────────────────────────────────────────────────────────────────
print('\n' + '='*80)
print('TOP-10 MOST DISCRIMINATIVE WORDS PER TOPIC CATEGORY')
print('='*80)

for topic, words in top_words_per_topic.items():

    print(f'\n🔹 Topic: {topic}')
    print('-'*50)

    for rank, (word, score) in enumerate(words, 1):
        print(f'{rank:2d}. {word:<20} {score:.4f}')


# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: Save Results
# ─────────────────────────────────────────────────────────────────────────────
with open('embeddings/top_words_per_topic.json', 'w', encoding='utf-8') as f:
    json.dump(top_words_per_topic, f, ensure_ascii=False, indent=4)

print('\nSaved → embeddings/top_words_per_topic.json')

Loaded metadata for 300 documents
Topic distribution:
Counter({'General': 24142, 'Crime': 32, 'Business': 30, 'Politics': 30, 'Sports': 4, 'Technology': 3})

TOP-10 MOST DISCRIMINATIVE WORDS PER TOPIC CATEGORY

🔹 Topic: Business
--------------------------------------------------
 1. ‘                    0.2012
 2. <UNK>                0.0848
 3. ذریعہ                0.0347
 4. ،تصویر               0.0339
 5. فیصد،                0.0311
 6. تارڑ                 0.0306
 7. عطا                  0.0284
 8. تھی                  0.0266
 9. دکھائی۔              0.0264
10. سبسکرائب             0.0252

🔹 Topic: Politics
--------------------------------------------------
 1. ‘                    0.1342
 2. ذریعہ                0.1040
 3. ،تصویر               0.1017
 4. کا                   0.0604
 5. نجکاری               0.0492
 6. ائی                  0.0478
 7. پی                   0.0386
 8. کی                   0.0336
 9. اے                   0.0329
10. ملازمین              0.0329

🔹 Topic: 